# Lesson 3, Exercise 1: Post-Training Quantization - Beyond Memory: Speed and Quality Trade-offs

**Goal:**
The primary goal of this exercise is to move beyond simply observing memory reduction from quantization and to comprehensively evaluate its impact. You will quantify and analyze the trade-offs between model memory footprint, inference speed (latency), and the subjective quality of generated text when applying different levels of Post-Training Quantization (PTQ) to a GPT-2 model.

## 2. Imports and Configuration

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import pandas as pd
import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)  # silence per-call "inputs will be cast" notices

MODEL_NAME = "gpt2" # Standard GPT-2
PROMPTS = [
    "The capital of France is",
    "Once upon a time, in a land far, far away,",
    "To be or not to be, that is the"
]
MAX_NEW_TOKENS = 50
NUM_TIMING_RUNS = 3 # Number of times to run generation for averaging latency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


## 3. Helper Functions

In [2]:
def get_model_memory_footprint(model):
    """Gets model memory footprint in MB."""
    # bytes = number of elements * bytes per element (works for int8/uint8 quantized weights too)
    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_bufs = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])
    mem = mem_params + mem_bufs # in bytes
    return mem / 1024**2 # convert to MB

def generate_text_and_time(model, tokenizer, prompt, max_new_tokens):
    """Generates text and returns the generated text and latency."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start_time = time.perf_counter() # Use perf_counter for more precise timing
    if model.device.type == 'cuda':
        torch.cuda.synchronize() # Ensure previous CUDA ops are done

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # greedy -> deterministic, comparable across precisions
            pad_token_id=tokenizer.pad_token_id,
        )

    if model.device.type == 'cuda':
        torch.cuda.synchronize() # Ensure generation is done
    end_time = time.perf_counter()

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    latency = end_time - start_time
    return generated_text, latency


## 4. Main Experiment Logic

In [3]:
results_list = [] # Use a different name to avoid conflict if re-running cells

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Where quantized weights should live. bitsandbytes needs a device_map; on CPU we pin everything to "cpu".
_qdev = "auto" if device.type == "cuda" else {"": "cpu"}

# Define precision configurations to test
configurations = [
    # Baseline: fp16 on GPU (bnb kernels also work in fp16), fp32 on CPU
    {"name": "FP16 (Baseline GPU)" if device.type == "cuda" else "FP32 (Baseline CPU)",
     "load_args": {"dtype": torch.float16 if device.type == "cuda" else torch.float32}},

    # INT8 (LLM.int8() weight-only quantization with outlier decomposition)
    {"name": "INT8 (bitsandbytes)",
     "load_args": {"quantization_config": BitsAndBytesConfig(load_in_8bit=True), "device_map": _qdev}},

    # NF4: 4-bit "normal float" data type, information-theoretically optimal for normally distributed weights
    {"name": "NF4 (bitsandbytes 4-bit)",
     "load_args": {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                             bnb_4bit_compute_dtype=torch.float16 if device.type == "cuda" else torch.float32),
                   "device_map": _qdev}},

    # FP4: plain 4-bit floating point
    {"name": "FP4 (bitsandbytes 4-bit)",
     "load_args": {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="fp4",
                                                             bnb_4bit_compute_dtype=torch.float16 if device.type == "cuda" else torch.float32),
                   "device_map": _qdev}},
    ]

# bitsandbytes historically required CUDA. Recent versions (>= 0.45 multi-backend, 0.50 here) also run
# int8/nf4/fp4 on CPU, so instead of filtering blindly we probe once and only drop the bnb configs if the
# CPU backend really is unavailable.
if device.type == "cpu":
    try:
        _probe = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": "cpu"})
        del _probe
        print("bitsandbytes CPU backend available - keeping INT8/NF4/FP4 configurations.")
    except Exception as e:
        print(f"bitsandbytes quantization not available on CPU ({type(e).__name__}). Filtering configurations.")
        configurations = [config for config in configurations if "bitsandbytes" not in config["name"]]

print(f"\n--- Starting Experiment for Model: {MODEL_NAME} ---")

for config in configurations:
    print(f"\nLoading model with configuration: {config['name']}")
    model = None # Ensure model is reset
    try:
        # Load the model. Quantized configs carry a device_map (required by bitsandbytes);
        # the plain baseline is loaded normally and moved to the device.
        current_load_args = dict(config['load_args'])
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **current_load_args)
        if "device_map" not in current_load_args:
            model.to(device)
        model.eval()

        if model is None: # Check if model loading was skipped/failed in TODO
            print(f"Skipping {config['name']} due to model loading not implemented in TODO.")
            continue

        # Memory footprint of parameters + buffers
        memory_mb = get_model_memory_footprint(model)
        print(f"Memory Footprint: {memory_mb:.2f} MB")

        avg_latencies_for_config = []
        generated_outputs_for_prompts = {}

        for i, prompt_text in enumerate(PROMPTS):
            print(f"  Processing prompt: '{prompt_text[:30]}...' ")
            prompt_specific_latencies = []
            current_generated_text = "N/A"

            if i == 0:
                # Warm-up (kernel/JIT setup, page-in of weights) so timing reflects steady state
                generate_text_and_time(model, tokenizer, prompt_text, 5)
                # Time the first prompt NUM_TIMING_RUNS times to get a stable latency estimate
                for run in range(NUM_TIMING_RUNS):
                    text, lat = generate_text_and_time(model, tokenizer, prompt_text, MAX_NEW_TOKENS)
                    prompt_specific_latencies.append(lat)
                    if run == 0:
                        current_generated_text = text
                avg_latencies_for_config.append(sum(prompt_specific_latencies) / len(prompt_specific_latencies))
                print(f"    Avg latency over {NUM_TIMING_RUNS} runs: {avg_latencies_for_config[-1]:.3f}s")
            else:
                # Other prompts: generate once, for quality assessment only
                current_generated_text, _ = generate_text_and_time(model, tokenizer, prompt_text, MAX_NEW_TOKENS)

            generated_outputs_for_prompts[f"Prompt {i+1} Output"] = current_generated_text

        overall_avg_latency_for_config = sum(avg_latencies_for_config) / len(avg_latencies_for_config) if avg_latencies_for_config else float('nan')

        results_list.append({
            "Float Precision": config["name"],
            "Memory (MB)": memory_mb,
            "Avg Latency (s)": overall_avg_latency_for_config, # Based on first prompt's timing
            **generated_outputs_for_prompts
        })

        del model # Free up memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"Could not run configuration {config['name']}. Error: {e}")
        results_list.append({
            "Float Precision": config["name"],
            "Memory (MB)": "Error",
            "Avg Latency (s)": "Error",
            **{f"Prompt {i+1} Output": "Error" for i in range(len(PROMPTS))}
        })

# --- Display Results ---
df_results = pd.DataFrame(results_list)
pd.set_option("display.max_colwidth", 120)
print("\n\n--- Experiment Results Summary ---")
print(df_results[["Float Precision", "Memory (MB)", "Avg Latency (s)"]].to_string())
for _, row in df_results.iterrows():
    print(f"\n[{row['Float Precision']}]")
    for i in range(len(PROMPTS)):
        print(f"  Prompt {i+1}: {row.get(f'Prompt {i+1} Output', '')!r}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

bitsandbytes CPU backend available - keeping INT8/NF4/FP4 configurations.

--- Starting Experiment for Model: gpt2 ---

Loading model with configuration: FP32 (Baseline CPU)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Memory Footprint: 474.70 MB
  Processing prompt: 'The capital of France is...' 


    Avg latency over 3 runs: 1.569s
  Processing prompt: 'Once upon a time, in a land fa...' 


...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Memory Footprint: 191.20 MB
  Processing prompt: 'The capital of France is...' 


    Avg latency over 3 runs: 4.470s
  Processing prompt: 'Once upon a time, in a land fa...' 


  Processing prompt: 'To be or not to be, that is th...' 




--- Experiment Results Summary ---
            Float Precision  Memory (MB)  Avg Latency (s)
0       FP32 (Baseline CPU)   474.700195         1.568752
1       INT8 (bitsandbytes)   231.700195         3.617190
2  NF4 (bitsandbytes 4-bit)   191.200195         3.855029
3  FP4 (bitsandbytes 4-bit)   191.200195         4.469707

[FP32 (Baseline CPU)]
  Prompt 1: 'The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.\n\nThe French Republic is the capital of the French Republic.\n\nThe French Republic is the capital of the French Republic.\n\n'
  Prompt 2: 'Once upon a time, in a land far, far away, the world was a land of the dead, and the dead were the living.\n\nThe dead were the living, and the living were the living.\n\nThe dead were the living, and the living were the living.\n\nThe dead'
  Prompt 3: 'To be or not to be, that is the question.\n\nThe question is, what is the difference between a "good"

## 5. Analysis and Discussion

*Run on CPU (Intel i7-10610U, 4 cores, AVX2), PyTorch 2.13, bitsandbytes 0.50.1 CPU backend, greedy decoding, 50 new tokens, latency = mean of 3 runs on prompt 1.*

| Precision | Memory (MB) | Avg latency (s) | vs FP32 |
|---|---|---|---|
| FP32 (baseline CPU) | 474.7 | 1.57 | 1.00× |
| INT8 (bitsandbytes LLM.int8) | 231.7 | 3.62 | 2.3× slower |
| NF4 (bitsandbytes 4‑bit) | 191.2 | 3.86 | 2.5× slower |
| FP4 (bitsandbytes 4‑bit) | 191.2 | 4.47 | 2.8× slower |

1.  **Memory Scaling:**
    *   The parameter+buffer footprint drops from **474.7 MB (fp32) → 231.7 MB (int8, −51 %) → 191.2 MB (4‑bit, −60 %)**. Only the `Linear` weights inside the transformer blocks are quantized; the token/position embeddings (≈ 40 M of GPT‑2's 124 M parameters), LayerNorms and the LM head remain in full precision, which is why int8 is close to the ideal 2× but 4‑bit is far from the ideal 8× — the un‑quantized 32‑bit embeddings (~160 MB) dominate the remaining footprint. On a larger model (Llama‑3.2‑1B, embeddings ≈ 20 % of params) the relative savings are much closer to the theoretical 2×/4×.

2.  **Latency Changes:**
    *   Latency did **not** decrease with lower precision — it got 2.3–2.8× *worse*. bitsandbytes' kernels dequantize the weights (int8/nf4/fp4 → fp16/fp32) on the fly inside every matmul; on a GPU that costs little relative to the memory‑bandwidth savings, but on this CPU backend the dequantization is done by generic (non‑AVX‑optimised) code and dominates. In addition, GPT‑2 small is tiny: at batch size 1 the per‑token cost is dominated by Python/framework overhead rather than weight bandwidth, so there is little bandwidth to save in the first place. Quantization is a *memory* optimisation first; latency wins only appear when (a) the model is bandwidth‑bound (large weights, small batch) and (b) fused low‑precision kernels exist for the hardware (e.g. Marlin/ExLlama on GPU, or `optimum-quanto`/`torch.ao` int8 kernels on CPU).

3.  **Output Quality Degradation:**
    *   **INT8** – outputs are almost token‑for‑token identical to fp32 (prompt 1 and 2 identical, prompt 3 diverges only after ~35 tokens). LLM.int8() with outlier decomposition is essentially lossless.
    *   **NF4** – still coherent and on‑topic but visibly different: prompt 2 changes from “a land of the dead” to “a place of great beauty and great danger”, and prompt 3 produces the semantically odd “the person who is good is the one who is bad”.
    *   **FP4** – clear degradation: prompt 1 becomes factually wrong (“The capital of France is the capital of the world”), prompt 3 loses the Shakespeare continuation entirely (“that is the way it is … the only way to be or not to be is to be or not to be”). Repetition loops also start earlier. Significant degradation therefore begins at 4 bits, and FP4 is noticeably worse than NF4 at the same bit‑width because NF4's quantile‑based bins match the (roughly Gaussian) weight distribution better.

4.  **Key Trade-offs:**
    *   *Memory:* int8 halves the model, 4‑bit gives ~60 % (more on larger models). *Speed:* on this CPU backend all quantized variants are slower; on CUDA int8 is roughly neutral and 4‑bit is often faster for bandwidth‑bound decoding. *Quality:* int8 ≈ lossless, NF4 minor drift, FP4 visible errors.
    *   **Best balance:** INT8 when the goal is to fit a model in half the memory with no quality loss (e.g. serving a model on a smaller GPU); **NF4** when memory is the hard constraint (fitting a 7B/13B model on a consumer GPU, QLoRA fine‑tuning) and a small quality hit is acceptable; **FP4** is hard to justify — NF4 costs the same and is better. **FP32/FP16** remains the right choice for tiny models like GPT‑2 on CPU where the footprint is already small and quantization only adds compute.
